# S-HIM / HIMLoco+AMP 在 Colab 上训练

1. 在 VS Code 安装 **Google.colab** 扩展，再打开本 Notebook。
2. 点击 **Select Kernel → Colab → L4**，登录 Google 账号。
3. 确认下方分支和实验配置，点击 **Run All**，并授权挂载 Google Drive。

In [ ]:
# 1. 配置实验
from pathlib import Path
import os
import signal
import subprocess


def stream_cmd(command, *, cwd=None, env=None):
    # 通过 Python stdout 将关键输出实时转发到 Notebook，过滤 4096 环境下的海量 Omniverse USD 重复警告防卡死浏览器。
    noisy = ("[Warning] [omni.", "[Warning] [isaacsim.", "[Error] [omni.ui]", "[Error] [omni.kit.renderer", "[Error] [carb.windowing")
    skipped = 0
    with subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, errors="replace", start_new_session=True,
    ) as process:
        try:
            for line in process.stdout:
                if any(tag in line for tag in noisy):
                    skipped += 1
                    continue
                print(line, end="", flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            process.communicate()
            raise
    if skipped:
        print(f"[stream_cmd] 已折叠 {skipped} 行 Omniverse 重复渲染/USD 警告（完整日志见 console.log）", flush=True)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)


REPO_URL = "https://github.com/qyw23AI/S-HIM-and-ES-WMP.git"
BRANCH = "main"
EXP_PRESET = "B1"  # "B1" = B1_himloco_amp_diag_4096, "L2" = L2_shim_spike_diag_4096, "L1" = L1_shim_nlif_diag_4096
SEED = 1
TOTAL_ITERATIONS = 5000
DRIVE_BASE = "/content/drive/MyDrive/shim_runs"
ENV_CACHE_PATH = "/content/drive/MyDrive/shim_cache/shim_env.tar.zst"
USE_ENV_CACHE = False
L4_CU_PER_HOUR = 3.0  # Colab L4 标准/高 RAM 约 2–5 CU/小时，请查看 Colab 显示的实时费率。

REPO = Path("/content/shim")
RUN_NAME = f"{EXP_PRESET}_seed{SEED}"
OUT = Path("/content/runs") / RUN_NAME
SYNC = Path(DRIVE_BASE) / RUN_NAME

In [ ]:
# 2. 检查 GPU、内存和磁盘，挂载 Drive
from google.colab import drive

stream_cmd(["nvidia-smi"])
stream_cmd(["free", "-h"])
stream_cmd(["df", "-h", "/content"])
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
if "L4" in gpu:
    print("L4：建议先跑 4096 环境测速，再开始长训。")
elif "T4" in gpu:
    print("T4：4096 环境可能显存不足。如遇 OOM，切换到 L4。")
elif "A100" in gpu:
    print("A100：缺少 Isaac Sim 5.1 要求的 RT Core，建议切换到 L4。")
drive.mount("/content/drive")

In [ ]:
# 3. 获取实验分支，固定本次运行的代码版本
if (REPO / ".git").is_dir():
    stream_cmd(
        ["git", "fetch", "--depth", "1", "--no-recurse-submodules", REPO_URL, BRANCH],
        cwd=REPO,
    )
    stream_cmd(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO)
else:
    stream_cmd(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
print("实验提交", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

In [ ]:
# 4. 安装用户态环境，可选 Drive 归档
setup = ["bash", "scripts/setup_shim_env.sh", "--skip-smoke"]
setup_env = os.environ.copy()
if USE_ENV_CACHE:
    setup_env["ENV_ARCHIVE"] = ENV_CACHE_PATH
    if not Path(ENV_CACHE_PATH).is_file():
        setup.append("--save-cache")
else:
    setup_env.pop("ENV_ARCHIVE", None)
stream_cmd(setup, cwd=REPO, env=setup_env)

In [ ]:
# 5. 4096 环境 × 5 轮测速，估算长训耗时与 L4 CU
import re
from statistics import median

def run_experiment(*args):
    command = ["bash", "scripts/run_shim_exp.sh", *map(str, args)]
    stream_cmd(command, cwd=REPO)

def read_metrics(log, keys):
    pattern = re.compile(
        r"'(" + "|".join(map(re.escape, keys)) + r")':\s*(None|nan|inf|-inf|[-+\d.eE]+)"
    )
    with log.open(errors="replace") as stream:
        for line in stream:
            if "'iteration':" in line:
                yield dict(pattern.findall(line))

smoke_out = Path("/content/runs") / f"smoke-{RUN_NAME}"
run_experiment(f"smoke-{EXP_PRESET}", "--out", smoke_out, "--seed", SEED)
durations = [
    float(row["iteration_seconds"])
    for row in read_metrics(smoke_out / "console.log", ("iteration_seconds",))
    if "iteration_seconds" in row
]
if not durations:
    raise RuntimeError("测速日志缺少 iteration_seconds")
seconds = median(durations[1:] or durations)
hours = seconds * TOTAL_ITERATIONS / 3600
print(f"每轮约 {seconds:.2f} 秒，{TOTAL_ITERATIONS} 轮约 {hours:.2f} 小时（训练迭代）")
if "L4" in gpu:
    print(f"L4 约 {hours * L4_CU_PER_HOUR:.1f} CU（按 {L4_CU_PER_HOUR:g} CU/小时估算）")

In [ ]:
# 6. 长训，定时同步并自动续跑
run_experiment(
    EXP_PRESET, "--out", OUT, "--sync-dir", SYNC,
    "--total-iterations", TOTAL_ITERATIONS, "--seed", SEED, "--resume-auto"
)

In [ ]:
# 7. 查看本地/Drive 最新指标与备份检查点
from collections import deque

keys = ("iteration", "terrain_level", "command_aligned_velocity_mps", "iteration_seconds", "learning_rate")
for label, directory in (("本地", OUT), ("Drive", SYNC)):
    log = directory / "console.log"
    if log.is_file():
        latest = deque(read_metrics(log, keys), maxlen=1)
        if latest:
            print(label, {key: latest[0].get(key, "—") for key in keys})
print("Drive 检查点", [path.name for path in sorted(SYNC.glob("model*.pt"))])